Análise exploratória dos arquivos originais com PySpark

Exploração para conhecer os dados antes da criação das camadas Bronze, Silver e Gold.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, mean, when

In [ ]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('analise_exploratoria_alfabetizacao')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('Versão do Spark:', spark.version)

In [ ]:
pasta_atual = Path.cwd()

if pasta_atual.name == 'notebooks':
    raiz_projeto = pasta_atual.parent
else:
    raiz_projeto = pasta_atual

pasta_raw = raiz_projeto / 'data' / 'arquivos_raw'

print('Raiz do projeto:', raiz_projeto)
print('Pasta dos arquivos originais:', pasta_raw)
print('A pasta existe?', pasta_raw.exists())

In [ ]:
arquivos_encontrados = sorted(pasta_raw.glob('*.csv.gz'))

for arquivo in arquivos_encontrados:
    tamanho_kb = arquivo.stat().st_size / 1024
    print(arquivo.name, '-', round(tamanho_kb, 2), 'KB')

print('Total de arquivos:', len(arquivos_encontrados))

In [ ]:
def carregar_arquivo(nome_arquivo):
    caminho = pasta_raw / nome_arquivo

    dataframe = (
        spark.read
        .option('header', True)
        .option('inferSchema', True)
        .option('encoding', 'UTF-8')
        .csv(caminho.as_posix())
    )

    return dataframe

In [ ]:
df_uf = carregar_arquivo('br_inep_avaliacao_alfabetizacao_uf.csv.gz')
df_meta_brasil = carregar_arquivo('br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv.gz')
df_meta_uf = carregar_arquivo('br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv.gz')
df_meta_municipio = carregar_arquivo('br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv.gz')
df_municipio = carregar_arquivo('br_inep_avaliacao_alfabetizacao_municipio.csv.gz')
df_alunos = carregar_arquivo('br_inep_avaliacao_alfabetizacao_aluno.csv.gz')

print('DataFrames carregados com sucesso.')

Quantidade de registros e colunas

In [ ]:
dataframes = {
    'uf': df_uf,
    'meta_brasil': df_meta_brasil,
    'meta_uf': df_meta_uf,
    'meta_municipio': df_meta_municipio,
    'municipio': df_municipio,
    'alunos': df_alunos,
}

for nome, dataframe in dataframes.items():
    linhas = dataframe.count()
    colunas = len(dataframe.columns)
    print(nome, '- linhas:', linhas, '- colunas:', colunas)

Primeiras linhas

In [ ]:
for nome, dataframe in dataframes.items():
    print('DataFrame:', nome)
    dataframe.show(5, truncate=False)

Esquema das tabelas

In [ ]:
for nome, dataframe in dataframes.items():
    print('Esquema:', nome)
    dataframe.printSchema()

Valores ausentes

In [ ]:
def mostrar_valores_nulos(dataframe):
    contagens = []

    for nome_coluna in dataframe.columns:
        contagem = count(when(col(nome_coluna).isNull(), nome_coluna)).alias(nome_coluna)
        contagens.append(contagem)

    dataframe.select(contagens).show(truncate=False)

In [ ]:
for nome, dataframe in dataframes.items():
    print('Valores nulos:', nome)
    mostrar_valores_nulos(dataframe)

Linhas duplicadas

In [ ]:
for nome, dataframe in dataframes.items():
    total = dataframe.count()
    total_sem_repeticao = dataframe.dropDuplicates().count()
    duplicados = total - total_sem_repeticao
    print(nome, '- linhas duplicadas:', duplicados)

Estatísticas dos resultados

In [ ]:
print('Estatísticas por UF:')
df_uf.select('taxa_alfabetizacao', 'media_portugues').summary().show()

print('Estatísticas por município:')
df_municipio.select('taxa_alfabetizacao', 'media_portugues').summary().show()

Distribuição por ano e rede

In [ ]:
(
    df_uf
    .groupBy('ano', 'rede')
    .agg(
        count('*').alias('quantidade_registros'),
        mean('taxa_alfabetizacao').alias('media_taxa_alfabetizacao'),
    )
    .orderBy('ano', 'rede')
    .show(truncate=False)
)

Visualização das metas

In [ ]:
print('Metas do Brasil:')
df_meta_brasil.show(truncate=False)

print('Exemplo das metas por UF:')
df_meta_uf.show(10, truncate=False)

print('Exemplo das metas por município:')
df_meta_municipio.show(10, truncate=False)

Exploração dos alunos

In [ ]:
print('Quantidade por presença:')
df_alunos.groupBy('presenca').count().orderBy('presenca').show()

print('Quantidade por situação de alfabetização:')
df_alunos.groupBy('alfabetizado').count().orderBy('alfabetizado').show()

print('Quantidade por rede:')
df_alunos.groupBy('rede').count().orderBy('rede').show()

Estatísticas dos alunos


In [ ]:
df_alunos.select('proficiencia', 'peso_aluno').summary().show()

In [ ]:
spark.stop()